# Factor Model: Step 3 - Winsorization of Returns

## Paleologo's Framework: Step 3 of 6

**Objective**: Identify and winsorize outliers in returns using a robust statistical method.

### Paleologo's Robust Winsorization Method (p. 131-132):

**The robust z-score formula:**

$$d_{i,t} = \frac{\log(1 + r_{i,t})}{\text{median}(|\log(1 + r_{i,t+1}) - \log(1 + r_{i,t-1})|)}$$

Where:
- $r_{i,t}$ = return of stock $i$ at time $t$
- Numerator: log return at time $t$
- Denominator: robust volatility measure (median of absolute differences in log returns)

**Winsorization rule:**
- If $|d_{i,t}| > \text{threshold}$, then winsorize the return
- Capped value: $\text{sign}(d_{i,t}) \times \text{threshold} \times \text{denominator}$

### Parameters:
- **Threshold**: 4 (moderate, institutional standard)
- **Lookback window**: 126 days (~6 months)
- **Minimum observations**: 60 days for stable median calculation

### Key Principles:
1. Security-level (time-series) winsorization - NOT cross-sectional
2. Uses robust statistics (median) instead of mean/std (which are sensitive to outliers)
3. Report all winsorized instances for review
4. Already filtered for liquid universe in Step 2

In [ ]:
# Import libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

# Set display options
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)
pd.set_option('display.float_format', lambda x: '%.6f' % x)

# Plotting style
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette('husl')

print("Libraries imported successfully")
print(f"Pandas version: {pd.__version__}")
print(f"NumPy version: {np.__version__}")

## 3.1 Load Estimation Universe Data

In [ ]:
# Load the estimation universe from Step 2
data_path = 'russell2000_estimation_universe_step2.parquet'

print("Loading estimation universe...")
df = pd.read_parquet(data_path)

print(f"✓ Data loaded successfully")
print(f"  Shape: {df.shape}")
print(f"  Index: {df.index.names}")
print(f"  Columns: {df.columns.tolist()}")
print(f"  Date range: {df.index.get_level_values(0).min()} to {df.index.get_level_values(0).max()}")
print(f"  Unique symbols: {df.index.get_level_values(1).nunique():,}")

print("\nFirst few rows:")
display(df.head(10))

## 3.2 Define Winsorization Parameters

In [ ]:
# Winsorization parameters (Paleologo's method)
THRESHOLD = 4                        # Robust z-score threshold (|d_i,t| > 4 gets winsorized)
LOOKBACK_WINDOW = 126                # 6 months for median calculation
MIN_OBSERVATIONS = 60                # Minimum observations required for robust median

print("="*80)
print("WINSORIZATION PARAMETERS")
print("="*80)
print(f"\nMethod: Paleologo's Robust Z-Score (p. 131-132)")
print(f"\nFormula:")
print(f"  d_i,t = log(1 + r_i,t) / median(|log(1 + r_i,t+1) - log(1 + r_i,t-1)|)")
print(f"\nParameters:")
print(f"  Threshold: {THRESHOLD} (if |d_i,t| > {THRESHOLD}, winsorize)")
print(f"  Lookback Window: {LOOKBACK_WINDOW} days (~6 months)")
print(f"  Minimum Observations: {MIN_OBSERVATIONS} days")
print(f"\nWinsorization Rule:")
print(f"  If |d_i,t| > {THRESHOLD}:")
print(f"    Winsorized return = sign(d_i,t) × {THRESHOLD} × robust_volatility")
print("\n" + "="*80)

## 3.3 Calculate Simple Returns

Calculate raw returns from closing prices.

In [ ]:
print("Calculating simple returns...")

# Extract close prices as a panel (date × symbol)
# Reset index to get date and symbol as columns, then pivot
close_panel = df['close'].unstack(level=1)  # Unstack symbol level

print(f"Close price panel shape: {close_panel.shape}")
print(f"  Dates: {len(close_panel)}")
print(f"  Symbols: {len(close_panel.columns)}")

# Calculate simple returns: r_i,t = (P_t - P_{t-1}) / P_{t-1}
returns = close_panel.pct_change()

print(f"\n✓ Simple returns calculated")
print(f"  Shape: {returns.shape}")
print(f"  Non-null values: {returns.notna().sum().sum():,}")
print(f"\nReturn Statistics (before winsorization):")
print(returns.stack().describe())

# Check for extreme values
extreme_count = ((returns.abs() > 1.0).sum().sum())  # Returns > 100%
print(f"\nReturns with |r| > 100%: {extreme_count:,} ({extreme_count/returns.notna().sum().sum()*100:.4f}%)")

## 3.4 Calculate Log Returns

Step 1 of Paleologo's formula: Calculate log(1 + r_i,t)

In [ ]:
print("Calculating log returns...")
print("Formula: log(1 + r_i,t) where r_i,t is the simple return\n")

# Calculate log returns: log(1 + r_i,t)
# This is equivalent to: log(P_t / P_{t-1})
log_returns = np.log(1 + returns)

# Replace inf/-inf with NaN (can occur if return = -1, i.e., price goes to 0)
log_returns = log_returns.replace([np.inf, -np.inf], np.nan)

print(f"✓ Log returns calculated")
print(f"  Shape: {log_returns.shape}")
print(f"  Non-null values: {log_returns.notna().sum().sum():,}")
print(f"\nLog Return Statistics:")
print(log_returns.stack().describe())

# Check for extreme log returns
extreme_log = ((log_returns.abs() > 0.69).sum().sum())  # log(2) ≈ 0.69 (i.e., 100% return)
print(f"\nLog returns with |log(1+r)| > 0.69 (>100%): {extreme_log:,}")

## 3.5 Calculate First Differences of Log Returns

Step 2 of Paleologo's formula: Calculate log(1 + r_i,t+1) - log(1 + r_i,t-1)

This measures the change in returns over a 2-day window.

In [ ]:
print("Calculating first differences of log returns...")
print("Formula: log(1 + r_i,t+1) - log(1 + r_i,t-1)")
print("This captures the 2-day change in log returns (second derivative of log prices)\n")

# Calculate: log(1 + r_i,t+1) - log(1 + r_i,t-1)
# This is equivalent to: log_returns[t+1] - log_returns[t-1]
# We shift forward (t+1) and backward (t-1) and take the difference

log_returns_forward = log_returns.shift(-1)    # log(1 + r_i,t+1)
log_returns_backward = log_returns.shift(1)    # log(1 + r_i,t-1)

# First difference: log(1 + r_i,t+1) - log(1 + r_i,t-1)
log_return_diff = log_returns_forward - log_returns_backward

print(f"✓ First differences calculated")
print(f"  Shape: {log_return_diff.shape}")
print(f"  Non-null values: {log_return_diff.notna().sum().sum():,}")
print(f"\nFirst Difference Statistics:")
print(log_return_diff.stack().describe())

## 3.6 Calculate Absolute First Differences

Step 3 of Paleologo's formula: |log(1 + r_i,t+1) - log(1 + r_i,t-1)|

In [ ]:
print("Calculating absolute first differences...")
print("Formula: |log(1 + r_i,t+1) - log(1 + r_i,t-1)|\n")

# Take absolute value of first differences
abs_log_return_diff = log_return_diff.abs()

print(f"✓ Absolute first differences calculated")
print(f"  Shape: {abs_log_return_diff.shape}")
print(f"  Non-null values: {abs_log_return_diff.notna().sum().sum():,}")
print(f"\nAbsolute First Difference Statistics:")
print(abs_log_return_diff.stack().describe())

## 3.7 Calculate Rolling Median (Robust Volatility)

Step 4 of Paleologo's formula: median(|log(1 + r_i,t+1) - log(1 + r_i,t-1)|)

This is the **denominator** of the robust z-score - a robust measure of volatility.

In [ ]:
print("Calculating rolling median of absolute first differences...")
print(f"Formula: median(|log(1 + r_i,t+1) - log(1 + r_i,t-1)|)")
print(f"Window: {LOOKBACK_WINDOW} days")
print(f"Minimum observations: {MIN_OBSERVATIONS} days")
print("\nThis is the DENOMINATOR of the robust z-score (robust volatility measure)\n")
print("This may take a few minutes...\n")

# Calculate rolling median for each stock (column-wise)
# Using rolling() with window and min_periods
robust_volatility = abs_log_return_diff.rolling(
    window=LOOKBACK_WINDOW,
    min_periods=MIN_OBSERVATIONS
).median()

print(f"✓ Robust volatility calculated")
print(f"  Shape: {robust_volatility.shape}")
print(f"  Non-null values: {robust_volatility.notna().sum().sum():,}")
print(f"  Null values: {robust_volatility.isna().sum().sum():,}")
print(f"\nRobust Volatility Statistics:")
print(robust_volatility.stack().describe())

# Check for zero volatility (problematic for division)
zero_vol_count = (robust_volatility == 0).sum().sum()
print(f"\nObservations with zero robust volatility: {zero_vol_count:,}")
if zero_vol_count > 0:
    print("  (These will be excluded from winsorization to avoid division by zero)")

## 3.8 Calculate Robust Z-Score (d_i,t)

**FINAL FORMULA:**

$$d_{i,t} = \frac{\log(1 + r_{i,t})}{\text{median}(|\log(1 + r_{i,t+1}) - \log(1 + r_{i,t-1})|)}$$

This is the complete Paleologo robust z-score.

In [ ]:
print("="*80)
print("CALCULATING ROBUST Z-SCORE (d_i,t)")
print("="*80)
print("\nFormula:")
print("  d_i,t = log(1 + r_i,t) / median(|log(1 + r_i,t+1) - log(1 + r_i,t-1)|)")
print("\nComponents:")
print("  Numerator: log(1 + r_i,t) - the log return at time t")
print("  Denominator: robust volatility (median of absolute first differences)")
print("\n" + "="*80 + "\n")

# Calculate robust z-score: d_i,t = log_returns / robust_volatility
# Replace zero volatility with NaN to avoid division by zero
robust_volatility_safe = robust_volatility.replace(0, np.nan)

# Calculate d_i,t
robust_zscore = log_returns / robust_volatility_safe

# Replace inf/-inf with NaN
robust_zscore = robust_zscore.replace([np.inf, -np.inf], np.nan)

print(f"✓ Robust z-scores calculated")
print(f"  Shape: {robust_zscore.shape}")
print(f"  Non-null values: {robust_zscore.notna().sum().sum():,}")
print(f"\nRobust Z-Score Statistics:")
print(robust_zscore.stack().describe())

# Analyze z-score distribution
z_abs = robust_zscore.abs().stack()
print(f"\nAbsolute Z-Score Distribution:")
print(f"  Mean: {z_abs.mean():.4f}")
print(f"  Median: {z_abs.median():.4f}")
print(f"  95th percentile: {z_abs.quantile(0.95):.4f}")
print(f"  99th percentile: {z_abs.quantile(0.99):.4f}")
print(f"  99.9th percentile: {z_abs.quantile(0.999):.4f}")
print(f"  Max: {z_abs.max():.4f}")

## 3.9 Identify Outliers

**Outlier Rule:** If |d_i,t| > threshold, mark as outlier for winsorization

In [ ]:
print("="*80)
print("IDENTIFYING OUTLIERS")
print("="*80)
print(f"\nThreshold: {THRESHOLD}")
print(f"Rule: If |d_i,t| > {THRESHOLD}, mark as outlier\n")

# Identify outliers: |d_i,t| > threshold
is_outlier = robust_zscore.abs() > THRESHOLD

# Count outliers
total_observations = robust_zscore.notna().sum().sum()
total_outliers = is_outlier.sum().sum()
outlier_pct = (total_outliers / total_observations) * 100 if total_observations > 0 else 0

print(f"Total observations with valid z-score: {total_observations:,}")
print(f"Outliers identified (|d_i,t| > {THRESHOLD}): {total_outliers:,} ({outlier_pct:.4f}%)")

# Breakdown by sign
positive_outliers = ((robust_zscore > THRESHOLD)).sum().sum()
negative_outliers = ((robust_zscore < -THRESHOLD)).sum().sum()

print(f"\nOutlier breakdown:")
print(f"  Positive outliers (d_i,t > {THRESHOLD}): {positive_outliers:,} ({positive_outliers/total_observations*100:.4f}%)")
print(f"  Negative outliers (d_i,t < -{THRESHOLD}): {negative_outliers:,} ({negative_outliers/total_observations*100:.4f}%)")

# Stocks with most outliers
outliers_per_stock = is_outlier.sum().sort_values(ascending=False)
print(f"\nTop 20 stocks with most outliers:")
print(outliers_per_stock.head(20))

## 3.10 Apply Winsorization

**Winsorization Formula:**

If |d_i,t| > threshold:
- Winsorized log return = sign(d_i,t) × threshold × robust_volatility
- Then convert back: winsorized simple return = exp(winsorized log return) - 1

In [ ]:
print("="*80)
print("APPLYING WINSORIZATION")
print("="*80)
print("\nWinsorization Formula:")
print(f"  If |d_i,t| > {THRESHOLD}:")
print(f"    Winsorized log return = sign(d_i,t) × {THRESHOLD} × robust_volatility")
print(f"    Winsorized simple return = exp(winsorized log return) - 1")
print("\n" + "="*80 + "\n")

# Initialize winsorized log returns (copy of original)
log_returns_winsorized = log_returns.copy()

# Calculate winsorized values for outliers
# Winsorized log return = sign(d_i,t) × threshold × robust_volatility
winsorized_values = np.sign(robust_zscore) * THRESHOLD * robust_volatility_safe

# Apply winsorization: replace outliers with winsorized values
log_returns_winsorized[is_outlier] = winsorized_values[is_outlier]

print(f"✓ Winsorization applied to log returns")
print(f"  Total values winsorized: {total_outliers:,}")

# Convert back to simple returns: r_winsorized = exp(log_return_winsorized) - 1
returns_winsorized = np.exp(log_returns_winsorized) - 1

# Replace inf/-inf with NaN
returns_winsorized = returns_winsorized.replace([np.inf, -np.inf], np.nan)

print(f"✓ Converted back to simple returns")
print(f"\nWinsorized Return Statistics:")
print(returns_winsorized.stack().describe())

# Compare original vs winsorized
print(f"\n" + "="*80)
print("COMPARISON: Original vs Winsorized Returns")
print("="*80)

original_stats = returns.stack().describe()
winsorized_stats = returns_winsorized.stack().describe()

comparison = pd.DataFrame({
    'Original': original_stats,
    'Winsorized': winsorized_stats,
    'Change': winsorized_stats - original_stats
})

print(comparison)

# Check extreme values before/after
extreme_original = (returns.abs() > 1.0).sum().sum()
extreme_winsorized = (returns_winsorized.abs() > 1.0).sum().sum()

print(f"\nExtreme returns (|r| > 100%):")
print(f"  Original: {extreme_original:,}")
print(f"  Winsorized: {extreme_winsorized:,}")
print(f"  Reduction: {extreme_original - extreme_winsorized:,}")

## 3.11 Visualize Distribution Changes

In [ ]:
print("Visualizing distribution changes...\n")

# Create comparison plots
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# Plot 1: Histogram comparison (zoomed in)
axes[0, 0].hist(returns.stack().dropna(), bins=100, alpha=0.5, label='Original', color='blue', range=(-0.5, 0.5))
axes[0, 0].hist(returns_winsorized.stack().dropna(), bins=100, alpha=0.5, label='Winsorized', color='red', range=(-0.5, 0.5))
axes[0, 0].set_title('Return Distribution Comparison (zoomed: -50% to +50%)', fontweight='bold', fontsize=12)
axes[0, 0].set_xlabel('Return')
axes[0, 0].set_ylabel('Frequency')
axes[0, 0].legend()
axes[0, 0].grid(True, alpha=0.3)

# Plot 2: Box plot comparison
data_to_plot = [
    returns.stack().dropna(),
    returns_winsorized.stack().dropna()
]
axes[0, 1].boxplot(data_to_plot, labels=['Original', 'Winsorized'])
axes[0, 1].set_title('Return Distribution - Box Plot', fontweight='bold', fontsize=12)
axes[0, 1].set_ylabel('Return')
axes[0, 1].grid(True, alpha=0.3)

# Plot 3: Q-Q plot style (percentiles)
percentiles = np.linspace(0, 100, 101)
original_percentiles = np.percentile(returns.stack().dropna(), percentiles)
winsorized_percentiles = np.percentile(returns_winsorized.stack().dropna(), percentiles)

axes[1, 0].plot(original_percentiles, winsorized_percentiles, 'o', alpha=0.5, markersize=4)
axes[1, 0].plot([-1, 1], [-1, 1], 'r--', linewidth=2, label='45° line (no change)')  # 45-degree line
axes[1, 0].set_title('Percentile Comparison: Original vs Winsorized', fontweight='bold', fontsize=12)
axes[1, 0].set_xlabel('Original Return Percentile')
axes[1, 0].set_ylabel('Winsorized Return Percentile')
axes[1, 0].legend()
axes[1, 0].grid(True, alpha=0.3)

# Plot 4: Tail comparison (extreme values)
extreme_threshold = 0.3  # 30%
original_tails = returns.stack().dropna()[returns.stack().dropna().abs() > extreme_threshold]
winsorized_tails = returns_winsorized.stack().dropna()[returns_winsorized.stack().dropna().abs() > extreme_threshold]

axes[1, 1].hist(original_tails, bins=50, alpha=0.5, label=f'Original (|r|>{extreme_threshold})', color='blue')
axes[1, 1].hist(winsorized_tails, bins=50, alpha=0.5, label=f'Winsorized (|r|>{extreme_threshold})', color='red')
axes[1, 1].set_title(f'Tail Distribution (|return| > {extreme_threshold*100:.0f}%)', fontweight='bold', fontsize=12)
axes[1, 1].set_xlabel('Return')
axes[1, 1].set_ylabel('Frequency')
axes[1, 1].legend()
axes[1, 1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 3.12 Generate Outlier Report

**Paleologo's Recommendation (p. 132):**
> "Make sure to report all the instances of winsorized data in backtest or in production, and examine them one by one."

In [ ]:
print("="*80)
print("OUTLIER REPORT - All Winsorized Instances")
print("="*80)
print("\nPaleologo's recommendation (p. 132):")
print('"Make sure to report all the instances of winsorized data')
print('in backtest or in production, and examine them one by one."')
print("\n" + "="*80 + "\n")

# Create outlier report DataFrame
# Get all outlier instances
outlier_mask_stacked = is_outlier.stack()
outlier_indices = outlier_mask_stacked[outlier_mask_stacked].index

if len(outlier_indices) > 0:
    outlier_report = pd.DataFrame({
        'date': [idx[0] for idx in outlier_indices],
        'symbol': [idx[1] for idx in outlier_indices],
        'original_return': [returns.loc[idx[0], idx[1]] for idx in outlier_indices],
        'winsorized_return': [returns_winsorized.loc[idx[0], idx[1]] for idx in outlier_indices],
        'log_return': [log_returns.loc[idx[0], idx[1]] for idx in outlier_indices],
        'robust_zscore': [robust_zscore.loc[idx[0], idx[1]] for idx in outlier_indices],
        'robust_volatility': [robust_volatility.loc[idx[0], idx[1]] for idx in outlier_indices]
    })
    
    # Calculate change
    outlier_report['return_change'] = outlier_report['winsorized_return'] - outlier_report['original_return']
    outlier_report['change_pct'] = (outlier_report['return_change'] / outlier_report['original_return'].abs()) * 100
    
    # Sort by absolute z-score (most extreme first)
    outlier_report = outlier_report.sort_values('robust_zscore', key=abs, ascending=False)
    
    print(f"Total winsorized instances: {len(outlier_report):,}\n")
    print("Top 50 most extreme outliers:")
    print("="*80)
    display(outlier_report.head(50))
    
    # Summary statistics
    print("\n" + "="*80)
    print("OUTLIER SUMMARY STATISTICS")
    print("="*80)
    print(f"\nOriginal returns (outliers only):")
    print(outlier_report['original_return'].describe())
    print(f"\nWinsorized returns (outliers only):")
    print(outlier_report['winsorized_return'].describe())
    print(f"\nRobust z-scores (outliers only):")
    print(outlier_report['robust_zscore'].describe())
    
    # Most affected stocks
    print("\n" + "="*80)
    print("STOCKS WITH MOST WINSORIZED INSTANCES")
    print("="*80)
    stock_outlier_counts = outlier_report['symbol'].value_counts().head(20)
    print(stock_outlier_counts)
    
    # Temporal distribution
    print("\n" + "="*80)
    print("TEMPORAL DISTRIBUTION OF OUTLIERS")
    print("="*80)
    outlier_report['year'] = pd.to_datetime(outlier_report['date']).dt.year
    yearly_outliers = outlier_report['year'].value_counts().sort_index()
    print("\nOutliers by year:")
    print(yearly_outliers)
    
else:
    print("No outliers detected!")
    outlier_report = pd.DataFrame()

In [ ]:
# Save outlier report to CSV for detailed examination
if len(outlier_report) > 0:
    outlier_report_path = 'winsorization_outlier_report.csv'
    outlier_report.to_csv(outlier_report_path, index=False)
    print(f"\n✓ Outlier report saved to: {outlier_report_path}")
    print(f"  Total outliers: {len(outlier_report):,}")
    print(f"\n  Examine each outlier to ensure winsorization was appropriate!")

## 3.13 Prepare Output Data

In [ ]:
print("="*80)
print("PREPARING OUTPUT DATA")
print("="*80)

# Add winsorized returns to the original dataframe
# Convert returns_winsorized back to long format
returns_winsorized_long = returns_winsorized.stack().to_frame(name='return_winsorized')

# Also add original returns for comparison
returns_long = returns.stack().to_frame(name='return_original')

# Add robust z-scores
robust_zscore_long = robust_zscore.stack().to_frame(name='robust_zscore')

# Add outlier flag
is_outlier_long = is_outlier.stack().to_frame(name='is_outlier')

# Merge all return data
df_output = df.copy()
df_output = df_output.join(returns_long, how='left')
df_output = df_output.join(returns_winsorized_long, how='left')
df_output = df_output.join(robust_zscore_long, how='left')
df_output = df_output.join(is_outlier_long, how='left')

print(f"✓ Output data prepared")
print(f"  Shape: {df_output.shape}")
print(f"  New columns added:")
print(f"    - return_original: original simple returns")
print(f"    - return_winsorized: winsorized simple returns")
print(f"    - robust_zscore: Paleologo's robust z-score (d_i,t)")
print(f"    - is_outlier: flag for winsorized observations")

print(f"\nColumns: {df_output.columns.tolist()}")
print(f"\nFirst few rows:")
display(df_output.head(10))

In [ ]:
# Save to file
output_path = 'russell2000_winsorized_step3.parquet'
print(f"\nSaving winsorized data to {output_path}...")

df_output.to_parquet(output_path, compression='snappy')

import os
file_size_mb = os.path.getsize(output_path) / 1024**2

print(f"✓ Data saved successfully!")
print(f"  File: {output_path}")
print(f"  Size: {file_size_mb:.2f} MB")
print(f"  Rows: {len(df_output):,}")
print(f"  Columns: {len(df_output.columns)}")

## 3.14 Summary Report

In [ ]:
print("="*80)
print("WINSORIZATION - SUMMARY REPORT")
print("="*80)

print(f"\nReport Date: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")

print("\n" + "="*80)
print("METHOD")
print("="*80)
print(f"\nPaleologo's Robust Winsorization (p. 131-132)")
print(f"\nFormula:")
print(f"  d_i,t = log(1 + r_i,t) / median(|log(1 + r_i,t+1) - log(1 + r_i,t-1)|)")
print(f"\nParameters:")
print(f"  Threshold: {THRESHOLD}")
print(f"  Lookback Window: {LOOKBACK_WINDOW} days")
print(f"  Minimum Observations: {MIN_OBSERVATIONS} days")

print("\n" + "="*80)
print("RESULTS")
print("="*80)
print(f"\nTotal observations: {total_observations:,}")
print(f"Outliers detected: {total_outliers:,} ({outlier_pct:.4f}%)")
print(f"  Positive outliers: {positive_outliers:,}")
print(f"  Negative outliers: {negative_outliers:,}")

print("\n" + "="*80)
print("IMPACT ON DISTRIBUTION")
print("="*80)

print(f"\nReturn Statistics:")
print(f"{'Metric':<20s} {'Original':>15s} {'Winsorized':>15s} {'Change':>15s}")
print("-" * 68)
metrics = ['mean', 'std', 'min', '25%', '50%', '75%', 'max']
for metric in metrics:
    orig_val = returns.stack().describe()[metric]
    wins_val = returns_winsorized.stack().describe()[metric]
    change = wins_val - orig_val
    print(f"{metric:<20s} {orig_val:>15.6f} {wins_val:>15.6f} {change:>15.6f}")

print(f"\nExtreme returns (|r| > 100%):")
print(f"  Before: {extreme_original:,}")
print(f"  After: {extreme_winsorized:,}")
print(f"  Reduction: {extreme_original - extreme_winsorized:,}")

print("\n" + "="*80)
print("OUTPUT FILES")
print("="*80)
print(f"\nWinsorized data: {output_path}")
print(f"  Size: {file_size_mb:.2f} MB")
print(f"  Rows: {len(df_output):,}")
print(f"  Columns: {len(df_output.columns)}")

if len(outlier_report) > 0:
    print(f"\nOutlier report: {outlier_report_path}")
    print(f"  Total outliers documented: {len(outlier_report):,}")
    print(f"  ⚠ IMPORTANT: Review each outlier to ensure appropriate winsorization")

print("\n" + "="*80)
print("✅ STEP 3 COMPLETE: WINSORIZATION APPLIED")
print("="*80)
print("\nNext Step: Step 4 - Loadings Generation (Calculate Alpha 101 & 191 factors)")